Statistical Engine
Build the core analysis logic first before touching any LLM:

Ingest a CSV with control/variant metrics
Calculate p-values, confidence intervals, relative lift
Flag underpowered tests (sample size checks)
Output a structured result dictionary

Tools: pandas, scipy.stats, numpy

In [3]:
import pandas as pd
import numpy as np
import scipy.stats as stats

In [4]:
df = pd.read_csv('../data/ab_test_experiments.csv')
df.head()

,test_id,test_name,product_area,hypothesis,primary_metric,secondary_metric,device_type,audience_segment,start_date,end_date,...,relative_lift_pct,secondary_metric_control,secondary_metric_variant,avg_order_value,control_revenue,variant_revenue,revenue_delta,outcome_type,analyst,notes
0,EXP-1000,Category Filter UX v1,Acquisition,Clearer value proposition will improve activation,signup_rate,bounce_rate,Mobile,Returning Users,2023-12-13,2023-12-26,...,9.64,0.2576,0.2335,119.45,2183307.10,2393300.20,209993.10,clear_winner,Chris P.,Test targeting returning users on mobile. Hypo...
1,EXP-1001,Mobile Nav Redesign v4,Retention,Reducing friction will increase conversion,signup_rate,avg_session_duration,Mobile,Free Tier,2023-05-23,2023-06-20,...,-15.31,38.0477,42.3042,311.77,5300713.54,2993615.54,-2307098.00,clear_loser,Sam T.,Test targeting free tier on mobile. Hypothesis...
2,EXP-1002,Email Subject Line Personalization v4,Acquisition,Simplified UI will reduce drop-off,activation_rate,pages_per_session,Tablet,All Users,2023-12-27,2024-01-15,...,-2.14,0.4291,0.4791,205.56,515338.92,504444.24,-10894.68,neutral,Alex R.,Test targeting all users on tablet. Hypothesis...
3,EXP-1003,Free Trial CTA vs Demo CTA v3,Onboarding,Social proof will increase trust and signups,signup_rate,avg_session_duration,Mobile,All Users,2023-03-25,2023-04-16,...,8.44,208.5381,231.6260,393.60,5252198.40,1423257.60,-3828940.80,novelty_effect,Heather B.,Test targeting all users on mobile. Hypothesis...
4,EXP-1004,Social Proof Placement v3,Acquisition,Simplified UI will reduce drop-off,application_completion,click_through_rate,Tablet,Returning Users,2023-07-22,2023-08-29,...,15.63,0.1387,0.1451,451.47,2326876.38,672690.30,-1654186.08,clear_winner,Alex R.,Test targeting returning users on tablet. Hypo...


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   test_id                    200 non-null    object 
 1   test_name                  200 non-null    object 
 2   product_area               200 non-null    object 
 3   hypothesis                 200 non-null    object 
 4   primary_metric             200 non-null    object 
 5   secondary_metric           200 non-null    object 
 6   device_type                200 non-null    object 
 7   audience_segment           200 non-null    object 
 8   start_date                 200 non-null    object 
 9   end_date                   200 non-null    object 
 10  test_duration_days         200 non-null    int64  
 11  traffic_split_control_pct  200 non-null    int64  
 12  control_visitors           200 non-null    int64  
 13  variant_visitors           200 non-null    int64  